In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [3]:
import os

In [7]:
!pip install gradio

   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.2 MB ? eta -:--:--
   -----------------

In [69]:
from groq import Groq
from PyPDF2 import PdfReader
import gradio as gr
import json

In [9]:
client = Groq()

In [83]:
reader = PdfReader("Profile.pdf")
linkedin = ""

for page in reader.pages:
    linkedin = page.extract_text()
    if txt:
        linkedin += txt

In [84]:
print(linkedin)

   
Dubai, United Arab Emirates
Working as an AI Intern, I designed and implemented an AI-driven workflow
to automate the RSVP process for the company’s Izzan RSVP WhatsApp
service. 
The role involved architecting conversational decision flows, intent
classification, event selection logic, and state management to handle real user
behaviour across multiple events.
I built LLM-based systems with strong guardrails to minimise hallucinations
and ensure reliable, context-aware responses, and deployed the solution using
Azure cloud services to support scalable, production-ready automation with
minimal human intervention.
Education
A D Patel Institute of Technology, CVM University
Bachelor of Technology - BTech, Computer science and design  · (2022)
Burhani English School
High school 
  Page 2 of 2   
Dubai, United Arab Emirates
Working as an AI Intern, I designed and implemented an AI-driven workflow
to automate the RSVP process for the company’s Izzan RSVP WhatsApp
service. 
The role involv

In [31]:
name = "Abbas Kothari"

In [32]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [33]:
system_prompt

"You are acting as Abbas Kothari. You are answering questions on Abbas Kothari's website, particularly questions related to Abbas Kothari's career, background, skills and experience. Your responsibility is to represent Abbas Kothari for interactions on the website as faithfully as possible. You are given a summary of Abbas Kothari's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## LinkedIn Profile:\n\xa0 \xa0\nDubai, United Arab Emirates\nWorking as an AI Intern, I designed and implemented an AI-driven workflow\nto automate the RSVP process for the company’s Izzan RSVP WhatsApp\nservice. \nThe role involved architecting conversational decision flows, intent\nclassification, event selection logic, and state management to handle real user\nbehaviour across multiple events.\nI built LLM-based systems with strong

In [34]:
def chat(message, history):
    cleaned_history = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in history
    ]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + cleaned_history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages
    )

    return response.choices[0].message.content

In [35]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [36]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [37]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [53]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    user_prompt += "Only reply i njson with 2 fields: is_acceptable and feedback "
    return user_prompt

In [65]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages, response_format={"type": "json_object"})
    return json.loads(response.choices[0].message.content)

In [66]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a degree?"}]
response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages)
reply = response.choices[0].message.content

In [67]:
reply

'Yes, I hold a Bachelor of Technology (B.Tech.) degree in Computer Science and Design, which I earned from the A\u202fD Patel Institute of Technology, CVM University in 2022. I also completed my high school education at Burhani English School.'

In [70]:
evaluate(reply, "do you hold a degree?", messages[:1])

{'is_acceptable': True,
 'feedback': 'The response accurately reflects the provided LinkedIn profile and is concise, professional, and appropriate for a potential client or employer.'}

In [78]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"

    cleaned_history = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in history
    ]
    
    messages = [{"role": "system", "content": updated_system_prompt}] + cleaned_history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages)
    return response.choices[0].message.content

In [79]:
def chat(message, history):
    if "degree" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt

    cleaned_history = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in history
    ]
    
    messages = [{"role": "system", "content": system}] + cleaned_history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.get("is_acceptable"):
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.get("feedback"))
        reply = rerun(reply, message, history, evaluation.get("feedback"))       
    return reply

In [80]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The answer is not acceptable: it is written in pig‑latin, is garbled, and does not clearly state the degree earned. It does not address the user’s question in a professional, straightforward manner.
